<div style="text-align: center; padding-top: 30px; padding-bottom: 10px;">

<h1 style="font-size: 2.8em; font-weight: 600; margin-bottom: 0.2em;">
IC-Weighted Ensemble — Global Neural Network
</h1>

<p style="font-size: 1.2em; color: gray; font-style: italic; margin-top: 0;">
Information-criterion weights over all candidate architectures.<br/>
Mean surface coloured by IC-weighted model uncertainty.
</p>

</div>

## 1. Packages, configuration and data

In [2]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import plotly.graph_objects as go
import yaml
from scipy.interpolate import griddata
from utils import create_pred_input, model_confidence_plot, add_histogram
from utils.miscelaneous import Find_data_file
from models import MultivariateModelGlobal as Model
from models.helper_functions.global_model import Prepare

os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(0)

2026-05-25 12:22:30.093439: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 12:22:30.189984: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-25 12:22:30.621024: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-25 12:22:31.046771: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779711751.317818    2919 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779711751.42

In [24]:
# ── User-facing knobs ────────────────────────────────────────────────────────
run_dir   = '/workspaces/Paper_1/runs/estimation/2026-04-23_08-47-33_global_IC_country_trends'
criterion = 'AIC'   # 'BIC' or 'AIC' — controls ranking, best model and weights
n_max     = 20      # cap the candidate set to the top-N by `criterion` (None = use all)
# ─────────────────────────────────────────────────────────────────────────────

criterion = criterion.upper()
assert criterion in ('BIC', 'AIC'), "criterion must be 'BIC' or 'AIC'"
ic_idx = 0 if criterion == 'BIC' else 1   # index into results[node]

with open(f'{run_dir}/.hydra/config.yaml', 'r') as fh:
    cfg = yaml.safe_load(fh)['instance']

print('Config loaded.')
print(f"  holdout={cfg['holdout']}, country_trends={cfg['country_trends']}, "
      f"data_end={cfg.get('data_end')}")
print(f"  criterion={criterion}")

Config loaded.
  holdout=0, country_trends=True, data_end=2024
  criterion=AIC


In [25]:
data = pd.read_excel(Find_data_file('MainData.xlsx'))
if cfg.get('data_end') is not None:
    data = data[data['Year'] <= cfg['data_end']]

growth, precip, temp = Prepare(data)
x_train = {0: temp, 1: precip}

mean_temp = np.nanmean(data['TempPopWeight'])
std_temp  = np.nanstd(data['TempPopWeight'])
mean_prec = np.nanmean(data['PrecipPopWeight'])
std_prec  = np.nanstd(data['PrecipPopWeight'])

pred_input, T_grid, P_grid = create_pred_input(
    mc=False, mean_T=mean_temp, std_T=std_temp,
    mean_P=mean_prec, std_P=std_prec, precip_capped=True
)

print(f'Data loaded: {data["Year"].min()}–{data["Year"].max()}, '
      f'{data["CountryCode"].nunique()} countries.')

Data loaded: 1961–2023, 196 countries.


## 2. Candidate model set and visual surfaces

In [27]:
results_raw = dict(np.load(f'{run_dir}/results.npy', allow_pickle=True).item())
results = {k: v for k, v in results_raw.items() if v is not None}

# Keep only nodes that have saved weight files
all_nodes = [
    node for node in results
    if os.path.exists(f'{run_dir}/parameters/{node}.weights.h5')
]

sorted_nodes = sorted(all_nodes, key=lambda n: results[n][ic_idx])

if n_max is not None:
    sorted_nodes = sorted_nodes[:n_max]

# Name → node-tuple lookup (needed later to index nn_surfaces)
node_by_name = {str(n): n for n in sorted_nodes}

print(f'Candidate set: {len(sorted_nodes)} models (ranked by {criterion})')
print(f'\nRank | Node                 | BIC           | AIC')
print('-' * 55)
for rank, node in enumerate(sorted_nodes, 1):
    bic, aic = results[node][0], results[node][1]
    print(f'{rank:4d} | {str(node):20s} | {bic:12.1f}  | {aic:.1f}')

Candidate set: 20 models (ranked by AIC)

Rank | Node                 | BIC           | AIC
-------------------------------------------------------
   1 | (4, 4, 2)            |     -52649.1  | -57667.9
   2 | (4, 2, 2)            |     -52739.7  | -57657.2
   3 | (2, 2, 2)            |     -52803.4  | -57648.4
   4 | (8, 2, 2)            |     -52581.9  | -57644.2
   5 | (8, 4, 4)            |     -52322.5  | -57631.1
   6 | (8, 2)               |     -52606.9  | -57625.8
   7 | (8, 4)               |     -52436.3  | -57600.0
   8 | (16, 2)              |     -52278.8  | -57587.4
   9 | (16, 2, 2)           |     -52226.3  | -57578.3
  10 | (4, 4)               |     -52595.6  | -57556.5
  11 | (8, 8, 2)            |     -52001.8  | -57542.1
  12 | (4, 4, 4)            |     -52430.1  | -57535.8
  13 | (16, 4, 2)           |     -51901.9  | -57529.1
  14 | (8, 8, 4)            |     -51834.2  | -57519.4
  15 | (4, 2)               |     -52639.4  | -57513.5
  16 | (2, 2)              

In [28]:
nn_surfaces = {}   # node-tuple → 2-D array for 3-D plotting

print('Loading models and computing visual surfaces ...')
for idx, node in enumerate(sorted_nodes, 1):
    weight_file = f'{run_dir}/parameters/{node}.weights.h5'

    factory = Model(node=node, cfg=cfg, x_train=x_train, y_train=growth)
    factory.Depth = len(node)
    model = factory.get_model()
    model.load_params(weight_file)

    pred_flat = model.model_visual.predict([pred_input], verbose=0).reshape(-1)
    nn_surfaces[node] = pred_flat.reshape(T_grid.shape)

    if idx % 5 == 0 or idx == len(sorted_nodes):
        print(f'  [{idx:3d}/{len(sorted_nodes)}] {str(node):20s}')

print('Done.')

Loading models and computing visual surfaces ...
  [  5/20] (8, 4, 4)           
  [ 10/20] (4, 4)              
  [ 15/20] (4, 2)              
  [ 20/20] (16, 4, 4)          
Done.


## 3. Burke benchmark — visual surface

In [29]:
d_burke = data.rename(columns={
    'TempPopWeight':  'temp',
    'PrecipPopWeight':'precip_raw',
    'GrowthWDI':      'growth',
    # CountryCode left as-is; ISO already exists in the data
}).copy()

d_burke['precip']   = d_burke['precip_raw'] / 1000
d_burke['temp_sq']  = d_burke['temp']   ** 2
d_burke['precip_sq']= d_burke['precip'] ** 2

yi_vars = [c for c in d_burke.columns if '_yi_' in c]
y2_vars = [c for c in d_burke.columns if '_y2_' in c]

if yi_vars:
    trend_str = ' + '.join(yi_vars + y2_vars)
    formula   = (f'growth ~ temp + temp_sq + precip + precip_sq '
                 f'+ C(ISO) + C(Year) + {trend_str}')
    print(f'Using pre-computed country trends: {len(yi_vars)} linear, '
          f'{len(y2_vars)} quadratic columns.')
else:
    d_burke['t_idx'] = d_burke.groupby('ISO')['Year'].transform(
        lambda x: (x - x.min()).astype(float)
    )
    d_burke['t_idx2'] = d_burke['t_idx'] ** 2
    formula = ('growth ~ temp + temp_sq + precip + precip_sq '
               '+ C(ISO) + C(Year) + C(ISO):t_idx + C(ISO):t_idx2')
    print('No pre-computed trend columns found; using C(ISO):t_idx interactions.')

d_fit = d_burke.dropna(subset=['growth', 'temp', 'precip']).copy()
burke_model = smf.ols(formula=formula, data=d_fit).fit()

print(f'Burke OLS: N={int(burke_model.nobs)}, '
      f'R²={burke_model.rsquared:.4f}, BIC={burke_model.bic:.1f}')

Using pre-computed country trends: 196 linear, 196 quadratic columns.
Burke OLS: N=10324, R²=0.2323, BIC=-24001.7


In [30]:
# Load the pre-computed Burke 3-D surface (used for visual comparison)
bench_data = pd.read_csv(Find_data_file('Benchmark/3d_results_no_outliers.csv'))
bench_Z = griddata(
    points=(bench_data['temp'].values, bench_data['precip_value'].values * 1000),
    values=bench_data['avg_prediction'].values,
    xi=(T_grid, P_grid),
    method='linear'
)
print('Burke visual surface interpolated.')

Burke visual surface interpolated.


## 4. IC weights

For each model $i$ the information-criterion weight is

$$w_i = \frac{\exp\!\bigl(-\tfrac{1}{2}\,\Delta_i\bigr)}
             {\displaystyle\sum_j \exp\!\bigl(-\tfrac{1}{2}\,\Delta_j\bigr)},
\quad \Delta_i = \mathrm{IC}_i - \min_j \mathrm{IC}_j,$$

where $\mathrm{IC}$ is whichever criterion (AIC or BIC) is selected at the top of the notebook.
Burke is excluded — its IC is far above all NN models, giving it effectively zero weight.

In [31]:
ic_values = np.array([results[n][ic_idx] for n in sorted_nodes])
delta     = ic_values - ic_values.min()   # Δ_IC ≥ 0 for all models
log_w     = -0.5 * delta
log_w    -= log_w.max()                   # shift for numerical stability
weights   = np.exp(log_w)
weights  /= weights.sum()

print(f'{criterion} weights computed.')
print(f'  Effective model count (1/Σw²): {1 / np.sum(weights**2):.1f}')
print('  Top-5 models by weight:')
for r in np.argsort(weights)[::-1][:5]:
    print(f'    {str(sorted_nodes[r]):20s}  {criterion}={ic_values[r]:.1f}  w={weights[r]:.4f}')

AIC weights computed.
  Effective model count (1/Σw²): 1.0
  Top-5 models by weight:
    (4, 4, 2)             AIC=-57667.9  w=0.9953
    (4, 2, 2)             AIC=-57657.2  w=0.0046
    (2, 2, 2)             AIC=-57648.4  w=0.0001
    (8, 2, 2)             AIC=-57644.2  w=0.0000
    (8, 4, 4)             AIC=-57631.1  w=0.0000


## 5. Summary table

In [32]:
rows = []
for rank, (node, w) in enumerate(zip(sorted_nodes, weights), 1):
    rows.append({
        f'Rank ({criterion})': rank,
        'Model':                str(node),
        'BIC':                  round(results[node][0], 1),
        'AIC':                  round(results[node][1], 1),
        f'{criterion} weight':  round(float(w), 6),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

 Rank (AIC)      Model      BIC      AIC  AIC weight
          1  (4, 4, 2) -52649.1 -57667.9    0.995339
          2  (4, 2, 2) -52739.7 -57657.2    0.004597
          3  (2, 2, 2) -52803.4 -57648.4    0.000058
          4  (8, 2, 2) -52581.9 -57644.2    0.000007
          5  (8, 4, 4) -52322.5 -57631.1    0.000000
          6     (8, 2) -52606.9 -57625.8    0.000000
          7     (8, 4) -52436.3 -57600.0    0.000000
          8    (16, 2) -52278.8 -57587.4    0.000000
          9 (16, 2, 2) -52226.3 -57578.3    0.000000
         10     (4, 4) -52595.6 -57556.5    0.000000
         11  (8, 8, 2) -52001.8 -57542.1    0.000000
         12  (4, 4, 4) -52430.1 -57535.8    0.000000
         13 (16, 4, 2) -51901.9 -57529.1    0.000000
         14  (8, 8, 4) -51834.2 -57519.4    0.000000
         15     (4, 2) -52639.4 -57513.5    0.000000
         16     (2, 2) -52694.5 -57496.1    0.000000
         17       (8,) -52564.1 -57496.0    0.000000
         18      (16,) -52316.2 -57480.0    0.

## 6. Visualisation

IC-weighted ensemble mean surface, coloured by IC-weighted standard deviation (model uncertainty).
The Burke OLS surface is overlaid in red for comparison.

In [33]:
surface_stack = np.stack([nn_surfaces[n] for n in sorted_nodes], axis=0)  # (M, H, W)
weighted_mean = np.einsum('m,mij->ij', weights, surface_stack)

# Center both surfaces at (mean_temp, mean_prec) so they pass through 0 there
temp_axis = T_grid[0, :]
prec_axis = P_grid[:, 0]
temp_idx  = int(np.argmin(np.abs(temp_axis - mean_temp)))
prec_idx  = int(np.argmin(np.abs(prec_axis - mean_prec)))

weighted_mean = weighted_mean - weighted_mean[prec_idx, temp_idx]
bench_Z       = bench_Z       - bench_Z[prec_idx, temp_idx]

print(f'Weighted mean — min={weighted_mean.min():.4f}, max={weighted_mean.max():.4f}')

fig = go.Figure()

fig.add_trace(go.Surface(
    x=T_grid, y=P_grid / 1000, z=weighted_mean,
    colorscale='Cividis',
    opacity=0.85,
    showscale=True,
    colorbar=dict(
        x=0.95, len=0.5,
        title=dict(text='Δ ln(GDP)', side='top')
    ),
    name=f'{criterion}-weighted NN ensemble'
))

fig.add_trace(go.Surface(
    x=T_grid, y=P_grid / 1000, z=bench_Z,
    colorscale='Reds', opacity=0.55,
    showscale=False,
    name='Burke OLS'
))

fig.update_layout(
    autosize=True,
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis_title='Temperature (°C)',
        yaxis_title='Precipitation (m)',
        zaxis=dict(title=dict(text='Δ ln(GDP)'), range=[-0.3, 0.3]),
        camera=dict(eye=dict(x=1.738, y=-1.780, z=0.589))
    ),
    showlegend=True,
    font=dict(size=10),
    annotations=[
        dict(
            text=f'{criterion}-weighted ensemble ({len(sorted_nodes)} NN models)',
            xref='paper', yref='paper',
            x=0.01, y=0.99,
            showarrow=False,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=11)
        )
    ]
)
fig.show()


Weighted mean — min=-0.2154, max=0.0268


In [14]:
fig.write_html('./results/images/3d_surf_ct.html')

In [34]:
temp_vals   = np.linspace(0, 30, 90)
precip_vals = np.linspace(12.03731002, 3000, 90)

prec_idx = int(np.argmin(np.abs(precip_vals - mean_prec)))
temp_idx = int(np.argmin(np.abs(temp_vals   - mean_temp)))

# IC-weighted mean curve at mean precipitation, centered at mean temperature
nn_curve    = weighted_mean[prec_idx, :].copy()
nn_curve   -= nn_curve[temp_idx]

# Burke curve at mean precipitation, centered at mean temperature
burke_curve  = bench_Z[prec_idx, :].copy()
burke_curve -= burke_curve[temp_idx]

fig2d = go.Figure()

fig2d.add_trace(go.Scatter(
    x=temp_vals, y=nn_curve,
    mode='lines',
    line=dict(width=2.5, color='steelblue'),
    name=f'{criterion}-weighted NN ensemble'
))

fig2d.add_trace(go.Scatter(
    x=temp_vals, y=burke_curve,
    mode='lines',
    line=dict(width=2.5, color='firebrick', dash='dash'),
    name='Quadratic OLS'
))

fig2d.add_hline(y=0, line=dict(color='black', width=1, dash='dot'))

fig2d.update_layout(
    xaxis_title='Temperature (°C)',
    yaxis_title='Δ ln(GDP)',
    legend=dict(bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1),
    font=dict(size=12),
    height=400
)
fig2d.show()

In [35]:
# Top-5 architectures by chosen criterion: curves at mean precipitation, centered at mean temperature
fig_top5 = go.Figure()

for node in sorted_nodes[:5]:
    curve = nn_surfaces[node][prec_idx, :].copy()
    curve -= curve[temp_idx]
    fig_top5.add_trace(go.Scatter(
        x=temp_vals, y=curve,
        mode='lines',
        line=dict(width=2),
        name=str(node)
    ))

fig_top5.add_hline(y=0, line=dict(color='black', width=1, dash='dot'))

fig_top5.update_layout(
    title=f'Top-5 architectures by {criterion}',
    xaxis_title='Temperature (°C)',
    yaxis_title='Δ ln(GDP)',
    legend=dict(bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1),
    font=dict(size=12),
    height=400
)
fig_top5.show()

## 7. Save outputs

In [3]:
outdir = './results/images/country_trends_vs_quad'

# summary_df.to_csv(f'{outdir}/aic_weights_summary.csv', index=False)
# np.save(f'{outdir}/aic_weighted_mean_surface.npy', weighted_mean)
# fig.write_html(f'{outdir}/aic_weighted_ensemble_3d.html')

fig2d.write_image(f'{outdir}/aic_weighted_ensemble_vs_Burke_2012.pdf', width=1600, height=1200, scale=2)


print(f'Outputs written to {outdir}/')

NameError: name 'fig2d' is not defined